In [ ]:
"""
Optimization of Boring Bar Dynamics
====================================
Rotating 5-section boring bar with integrated absorber.
FEM-based modal analysis + Budak-Altintas stability lobes.
Single design parameter: cavity axial position z_cav_start.

Geometry (5 sections, total L = 300 mm):
  Sec 1: Hollow cylinder, D_out=60mm, D_in=d_body_in, length L1
  Sec 2: Tapered outer, D decreases 60→40mm, D_in same, length L2
  Sec 3: Hollow cylinder, D=40mm, D_in=d_body_in, length L3
  Sec 4: Hollow, D=40mm, LARGER inner bore D4_in to house absorber, length L4
  Sec 5: Solid cylinder, D=40mm, length L5
  (D_sec3, D_sec4_out, D_sec5 are all 40mm)

Material:
  Body:   E=280 GPa, rho=rho_steel*1.2=9420 kg/m³
  Absorber mass: tungsten carbide, rho_tc=15600 kg/m³
  Damper oil: zeta_d=0.3

Cutting (boring, single-point):
  Nt=1, Ktc=2000 MPa, Krc=800 MPa (steel boring)
"""

import numpy as np
from scipy.linalg import eigh
from scipy.optimize import brentq, minimize_scalar
from scipy.integrate import quad
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size':9,'axes.titlesize':9,'axes.labelsize':9,
                     'xtick.labelsize':8,'ytick.labelsize':8,'figure.dpi':150})

In [ ]:
# ═══════════════════════════════════════════════════════
# 1. GLOBAL FIXED PARAMETERS
# ═══════════════════════════════════════════════════════

# Material
E_body   = 280e9          # Pa  (WC-composite body)
rho_body = 7850.0 * 1.2   # kg/m³
rho_tc   = 15600.0        # kg/m³  tungsten-carbide absorber

# Overall dimensions
L_total  = 0.300          # m   total length
D1_out   = 0.060          # m   Sec1 outer diameter
D_main   = 0.040          # m   Sec2-5 outer diameter (3,4,5 same)
d_body_wall = 0.003       # m   minimum wall thickness for hollow sections
D_body_in   = D_main - 2*d_body_wall   # inner bore of hollow body = 34 mm

# Section lengths (fixed proportions, total = L_total)
L1 = 0.050   # Sec1 large hollow cylinder
L2 = 0.040   # Sec2 taper
L3 = 0.080   # Sec3 hollow cylinder (before cavity)
L4 = 0.060   # Sec4 absorber cavity section  ← start position is the design variable
L5 = 0.070   # Sec5 solid cylinder (free end / tool end)
# Note: L1+L2+L3+L4+L5 = 0.300 ✓

# Absorber cavity (Sec4): larger inner bore
# absorber mass fills ≤70% cavity volume
# inner diameter of cavity
D4_in    = D_main - 2*d_body_wall  # will be updated per z_cav: we vary cavity start
# For the optimization the cavity length L4=60mm is fixed; only its START z_cav_start varies

# Absorber parameters (will be computed from cavity geometry)
zeta_d   = 0.3            # oil damping ratio

# Cutting parameters — boring, single-point
Nt   = 1
Ktc  = 2000e6   # N/m²
Krc  =  800e6   # N/m²
ratio_kr = Krc / Ktc
zeta_struct = 0.01        # structural damping ratio of bar body

# Speed range
RPM_max  = 6000           # boring bars typically < 6000 RPM
N_FEM    = 80             # elements per beam model
N_modes  = 4              # modes to retain

print("="*55)
print("Boring Bar FEM — fixed parameters loaded")
print(f"  Body: E={E_body/1e9:.0f} GPa, rho={rho_body:.0f} kg/m³")
print(f"  D1={D1_out*1e3:.0f}mm, D_main={D_main*1e3:.0f}mm, L={L_total*1e3:.0f}mm")
print(f"  Sections: {[L1,L2,L3,L4,L5]} m  sum={sum([L1,L2,L3,L4,L5]):.3f}m")
print("="*55)

In [ ]:
# ═══════════════════════════════════════════════════════
# 2. CROSS-SECTION PROPERTIES ALONG Z
# ═══════════════════════════════════════════════════════

def section_props(z, z_cav_start):
    """
    Returns (A, I, Ip, rhoA, rhoI, rhoIp) at axial position z
    given cavity start position z_cav_start.
    Sec boundaries depend on z_cav_start:
      Sec1: [0, L1]
      Sec2: [L1, L1+L2]   (taper)
      Sec3: [L1+L2, z_cav_start]
      Sec4: [z_cav_start, z_cav_start+L4]
      Sec5: [z_cav_start+L4, L_total]
    """
    z1 = L1
    z2 = L1 + L2
    z3 = z_cav_start          # Sec3 ends, Sec4 starts
    z4 = z_cav_start + L4     # Sec4 ends, Sec5 starts

    # Outer diameter at z (tapered in Sec2)
    if z < z1:
        D_out = D1_out
    elif z < z2:
        t = (z - z1) / L2     # 0→1
        D_out = D1_out + t*(D_main - D1_out)
    else:
        D_out = D_main

    R_out = D_out / 2

    # Inner diameter
    if z < z2:
        # Sec1 & taper: hollow with body wall
        R_in = max(0, R_out - d_body_wall)
    elif z < z3:
        # Sec3: hollow, inner = D_body_in/2
        R_in = D_body_in / 2
    elif z < z4:
        # Sec4: larger inner bore for absorber
        # Wall thickness must be ≥ 3mm
        R_in = R_out - d_body_wall   # = 17mm inner radius (D4_in/2)
    else:
        # Sec5: solid
        R_in = 0.0

    # Cross-section properties
    A  = np.pi * (R_out**2 - R_in**2)
    I  = np.pi * (R_out**4 - R_in**4) / 4
    Ip = np.pi * (R_out**4 - R_in**4) / 2

    rhoA  = rho_body * A
    rhoI  = rho_body * I
    rhoIp = rho_body * Ip

    return A, I, Ip, rhoA, rhoI, rhoIp

In [ ]:
# ═══════════════════════════════════════════════════════
# 3. ABSORBER GEOMETRY → mass, location
# ═══════════════════════════════════════════════════════

def absorber_params(z_cav_start):
    """
    Compute absorber mass from cavity geometry (70% fill, TC density).
    Absorber centre of mass = z_cav_start + L4/2.
    Returns (md, z_d)
    """
    R_out = D_main / 2
    R_in_body = D_body_in / 2
    R_cav = R_out - d_body_wall   # inner radius of cavity

    # Cavity volume (annular cylinder, 70% fill)
    V_cav  = np.pi * (R_cav**2 - 0**2) * L4  # full cavity bore
    # Actually absorber is solid plug inside cavity bore
    V_absorber = 0.70 * np.pi * R_cav**2 * L4
    md = rho_tc * V_absorber
    z_d = z_cav_start + L4 / 2  # absorber CG
    return md, z_d

In [ ]:
# ═══════════════════════════════════════════════════════
# 4. EULER-BERNOULLI FEM — Hermite cubic elements
# ═══════════════════════════════════════════════════════
# DOFs per node: [v, theta]  (transverse + rotation) in ONE plane
# Full 3D: assemble for x and y planes separately, then couple via
# gyroscopic/Coriolis terms.

def build_FEM_matrices(z_cav_start, Omega=0.0, with_absorber=False,
                       kd=0.0, cd=0.0):
    """
    Build global mass M, stiffness K, damping D (4-DOF system per
    transverse plane → 4n_nodes total after BC).

    State vector: [u_nodes(x-plane), v_nodes(y-plane)]
    Returns M, C, K as (2*n_free x 2*n_free) matrices.
    Also returns (PSI_zd, PSI_zf) modal participation (physical DOF indices).

    Uses 3-point Gauss integration for element matrices.
    """
    n_elem = N_FEM
    n_nodes = n_elem + 1
    L_e = L_total / n_elem
    z_nodes = np.linspace(0, L_total, n_nodes)

    # Global matrices — 2 DOF/node (v, theta) per plane, 2 planes
    dof = 2 * n_nodes   # per plane
    Mxx = np.zeros((dof, dof))
    Kxx = np.zeros((dof, dof))
    Gyro = np.zeros((dof, dof))   # gyroscopic/Coriolis coupling x↔y

    # Gauss points & weights (3-point)
    gp  = np.array([-np.sqrt(3/5), 0, np.sqrt(3/5)])
    gw  = np.array([5/9, 8/9, 5/9])

    for e in range(n_elem):
        z_a = z_nodes[e]
        z_b = z_nodes[e+1]
        le  = z_b - z_a

        # Element stiffness & mass (Hermite shape functions)
        Ke = np.zeros((4,4))
        Me = np.zeros((4,4))
        Ge = np.zeros((4,4))   # gyroscopic per element (polar inertia term)

        for g, w in zip(gp, gw):
            xi = (g + 1) / 2   # map [-1,1] → [0,1]
            z_g = z_a + xi*le
            _, I_g, Ip_g, rhoA_g, rhoI_g, rhoIp_g = section_props(z_g, z_cav_start)

            # Hermite shape functions N1..N4 and 2nd derivatives N1''..N4''
            N1  =  1 - 3*xi**2 + 2*xi**3
            N2  =  le*(xi - 2*xi**2 + xi**3)
            N3  =  3*xi**2 - 2*xi**3
            N4  =  le*(-xi**2 + xi**3)

            dN1 = (-6*xi + 6*xi**2)/le
            dN2 = (1 - 4*xi + 3*xi**2)
            dN3 = (6*xi - 6*xi**2)/le
            dN4 = (-2*xi + 3*xi**2)

            d2N1 = (-6 + 12*xi)/le**2
            d2N2 = (-4 + 6*xi)/le
            d2N3 = ( 6 - 12*xi)/le**2
            d2N4 = (-2 + 6*xi)/le

            Nv  = np.array([N1, N2, N3, N4])
            d2N = np.array([d2N1, d2N2, d2N3, d2N4])
            dN  = np.array([dN1, dN2, dN3, dN4])

            jac = le / 2 * 2   # dx = le*dxi,  integral weight = w*le/2*2 (from [-1,1]→[0,le])
            dz  = le * w / 2   # correct: integral(f dz) → sum w_g * f(xi_g) * le/2

            # Stiffness: EI * N'' ⊗ N''
            Ke += E_body * I_g * np.outer(d2N, d2N) * le * w / 2

            # Mass: rhoA * N ⊗ N  + rhoI * N' ⊗ N'
            Me += (rhoA_g * np.outer(Nv, Nv) + rhoI_g * np.outer(dN, dN)) * le * w / 2

            # Gyroscopic: rhoIp * N' ⊗ N'
            Ge += rhoIp_g * np.outer(dN, dN) * le * w / 2

        # Assemble into global (2*n_nodes x 2*n_nodes):
        # plane x: rows/cols [2e, 2e+1, 2e+2, 2e+3]
        idx = [2*e, 2*e+1, 2*(e+1), 2*(e+1)+1]
        for i, ii in enumerate(idx):
            for j, jj in enumerate(idx):
                Mxx[ii, jj] += Me[i, j]
                Kxx[ii, jj] += Ke[i, j]
                Gyro[ii, jj] += Ge[i, j]

    # Centrifugal softening (diagonal, acts as negative stiffness on v)
    # Kc_ii = -Omega^2 * rhoA * N⊗N (centrifugal)
    Kc = np.zeros((dof, dof))
    for e in range(n_elem):
        z_a = z_nodes[e]; z_b = z_nodes[e+1]; le = z_b - z_a
        Kce = np.zeros((4,4))
        for g, w in zip(gp, gw):
            xi = (g+1)/2; z_g = z_a + xi*le
            _, _, _, rhoA_g, _, _ = section_props(z_g, z_cav_start)
            N1 = 1-3*xi**2+2*xi**3; N2=le*(xi-2*xi**2+xi**3)
            N3 = 3*xi**2-2*xi**3;   N4=le*(-xi**2+xi**3)
            Nv = np.array([N1,N2,N3,N4])
            Kce += rhoA_g * np.outer(Nv,Nv) * le*w/2
        idx = [2*e,2*e+1,2*(e+1),2*(e+1)+1]
        for i,ii in enumerate(idx):
            for j,jj in enumerate(idx):
                Kc[ii,jj] += Kce[i,j]

    # Apply clamped BC at z=0: remove DOFs 0,1 (v=0, theta=0)
    free = list(range(2, dof))   # free DOFs per plane
    n_free = len(free)

    Mxx_f = Mxx[np.ix_(free, free)]
    Kxx_f = Kxx[np.ix_(free, free)]
    Gyro_f = Gyro[np.ix_(free, free)]
    Kc_f   = Kc[np.ix_(free, free)]

    # Structural damping (proportional to M)
    # C = alpha*M such that zeta = alpha/(2*wn) → alpha = 2*zeta*wn1
    # We'll add speed-independent structural damping after eigenvalue solve.

    # ---- Full 2-plane system with coupling ----
    # State: [u_free, v_free], size 2*n_free
    n2 = 2 * n_free
    M_full = np.zeros((n2, n2))
    K_full = np.zeros((n2, n2))
    C_full = np.zeros((n2, n2))   # gyroscopic + structural

    # Diagonal blocks: same Mxx, Kxx for both planes (isotropic)
    M_full[:n_free, :n_free] = Mxx_f
    M_full[n_free:, n_free:] = Mxx_f

    K_eff = Kxx_f - Omega**2 * Kc_f
    K_full[:n_free, :n_free] = K_eff
    K_full[n_free:, n_free:] = K_eff

    # Gyroscopic + Coriolis coupling (off-diagonal, skew-symmetric)
    G_eff = 2 * Omega * Gyro_f
    C_full[:n_free, n_free:] =  G_eff   # u-plane damped by v-velocity
    C_full[n_free:, :n_free] = -G_eff

    # Structural damping (we need wn1 — use zero-speed eigenvalue)
    evals, _ = eigh(Kxx_f, Mxx_f, subset_by_index=[0,0])
    wn1_approx = np.sqrt(max(evals[0], 1.0))
    alpha_struct = 2 * zeta_struct * wn1_approx
    C_struct = alpha_struct * M_full
    C_full += C_struct

    # ── Absorber augmentation ──────────────────────────────────────
    if with_absorber and kd > 0:
        md, z_d = absorber_params(z_cav_start)
        # Find node nearest z_d
        node_d = int(round(z_d / L_total * n_elem))
        node_d = min(node_d, n_elem)
        # Physical DOF index in free list: node displacement (even DOFs = v, odd = theta)
        dof_d_x = 2*node_d - 2   # index in free list (v-dof of node_d)
        dof_d_y = dof_d_x        # same index in y-plane
        # PSI at z_d: displacement DOF in free system
        # For absorber coupling we use translational DOF only

        # Augmented system: add absorber DOFs [ud, vd]
        # New size: (2*n_free + 2) x (2*n_free + 2)
        n_aug = n2 + 2
        Ma = np.zeros((n_aug, n_aug))
        Ka = np.zeros((n_aug, n_aug))
        Ca = np.zeros((n_aug, n_aug))

        Ma[:n2, :n2] = M_full
        Ka[:n2, :n2] = K_full
        Ca[:n2, :n2] = C_full

        # Absorber mass blocks
        Ma[n2,   n2]   = md
        Ma[n2+1, n2+1] = md

        # Absorber Coriolis (2*Omega*md coupling between ud and vd)
        Ca[n2,   n2+1] =  2*Omega*md
        Ca[n2+1, n2]   = -2*Omega*md

        # Spring/damper coupling: k_d*(u_beam(z_d) - u_d)^2
        # x-plane: dof_d_x in free list
        if dof_d_x < n_free:
            Ka[dof_d_x, dof_d_x] += kd
            Ka[dof_d_x, n2]       -= kd
            Ka[n2, dof_d_x]       -= kd
            Ka[n2, n2]             += kd - Omega**2*md

            Ca[dof_d_x, dof_d_x] += cd
            Ca[dof_d_x, n2]       -= cd
            Ca[n2, dof_d_x]       -= cd
            Ca[n2, n2]             += cd

        # y-plane: offset by n_free
        dof_d_y_global = n_free + dof_d_x
        if dof_d_x < n_free:
            Ka[dof_d_y_global, dof_d_y_global] += kd
            Ka[dof_d_y_global, n2+1]            -= kd
            Ka[n2+1, dof_d_y_global]            -= kd
            Ka[n2+1, n2+1]                       += kd - Omega**2*md

            Ca[dof_d_y_global, dof_d_y_global] += cd
            Ca[dof_d_y_global, n2+1]            -= cd
            Ca[n2+1, dof_d_y_global]            -= cd
            Ca[n2+1, n2+1]                       += cd

        return Ma, Ca, Ka, n_free, node_d, wn1_approx

    return M_full, C_full, K_full, n_free, -1, wn1_approx

In [ ]:
# ═══════════════════════════════════════════════════════
# 5. FIRST NATURAL FREQUENCY (zero speed)
# ═══════════════════════════════════════════════════════

def get_fn1(z_cav_start):
    M, C, K, n_free, _, _ = build_FEM_matrices(z_cav_start, 0.0, False)
    n = M.shape[0]//2
    evals, _ = eigh(K[:n,:n], M[:n,:n], subset_by_index=[0,0])
    return np.sqrt(max(evals[0],1.0)) / (2*np.pi)

In [ ]:
# ═══════════════════════════════════════════════════════
# 6. FRF COMPUTATION (physical space, tip DOF)
# ═══════════════════════════════════════════════════════

def compute_FRF_FEM(freqs_hz, z_cav_start, Omega=0.0,
                    with_absorber=False, kd=0.0, cd=0.0):
    """
    Returns Hxx, Hxy at free end node (tip).
    Force applied at tip, response at tip.
    """
    res = build_FEM_matrices(z_cav_start, Omega, with_absorber, kd, cd)
    M, C_mat, K = res[0], res[1], res[2]
    n_free = res[3]

    # Tip node = last node = node n_elem; free DOF = 2*(n_elem-1) translational
    tip_dof_x = 2*(N_FEM - 1)   # in free-dof indexing
    tip_dof_y = n_free + tip_dof_x

    n = M.shape[0]
    Hxx = np.zeros(len(freqs_hz), dtype=complex)
    Hxy = np.zeros(len(freqs_hz), dtype=complex)

    # Force vectors
    fx = np.zeros(n); fx[tip_dof_x] = 1.0
    fy = np.zeros(n); fy[tip_dof_y] = 1.0

    for k, f in enumerate(freqs_hz):
        w = 2*np.pi*f
        Z = -w**2 * M + 1j*w * C_mat + K
        try:
            sol = np.linalg.solve(Z, np.column_stack([fx, fy]))
            Hxx[k] = sol[tip_dof_x, 0]
            Hxy[k] = sol[tip_dof_x, 1]
        except:
            pass
    return Hxx, Hxy


In [ ]:
# ═══════════════════════════════════════════════════════
# 7. DIRECTIONAL COEFFICIENTS (boring: continuous cut)
# ═══════════════════════════════════════════════════════

def directional_boring():
    """
    Boring: single tooth, continuous 360° engagement.
    phi1=0, phi2=2*pi.  Altintas 2012 averaging.
    """
    phi = np.linspace(0, 2*np.pi, 100000)
    fx_fac = -np.cos(phi) - ratio_kr * np.sin(phi)
    fy_fac =  np.sin(phi) - ratio_kr * np.cos(phi)
    axx = np.trapezoid(np.sin(phi) * fx_fac, phi) / (2*np.pi)
    axy = np.trapezoid(np.cos(phi) * fx_fac, phi) / (2*np.pi)
    ayx = np.trapezoid(np.sin(phi) * fy_fac, phi) / (2*np.pi)
    ayy = np.trapezoid(np.cos(phi) * fy_fac, phi) / (2*np.pi)
    return axx, axy, ayx, ayy

axx_b, axy_b, ayx_b, ayy_b = directional_boring()
print(f"Boring dir. coeffs: axx={axx_b:.4f}, axy={axy_b:.4f}, "
      f"ayx={ayx_b:.4f}, ayy={ayy_b:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════
# 8. STABILITY LIMIT (scalar, for optimization)
# ═══════════════════════════════════════════════════════

def stability_limit_FRF(Hxx, Hxy, freqs_hz):
    """Min alim over chatter frequencies (Budak-Altintas)."""
    Hyy = Hxx; Hyx = Hxy
    alim_min = np.inf
    for k in range(len(Hxx)):
        h11 = axx_b*Hxx[k]+axy_b*Hyx[k]
        h12 = axx_b*Hxy[k]+axy_b*Hyy[k]
        h21 = ayx_b*Hxx[k]+ayy_b*Hyx[k]
        h22 = ayx_b*Hxy[k]+ayy_b*Hyy[k]
        tr  = h11+h22
        det = h11*h22 - h12*h21
        disc = tr**2 - 4*det
        sq = np.sqrt(disc+0j)
        for lam in [(-tr+sq)/(2*det+1e-60), (-tr-sq)/(2*det+1e-60)]:
            rl = np.real(lam); il = np.imag(lam)
            if rl < 0:
                al = (-2*np.pi*rl/(Nt*Ktc))*(1+(il/(rl+1e-60))**2)
                if 0 < al < alim_min:
                    alim_min = al
    return alim_min if alim_min < np.inf else 0.0

In [ ]:
# ═══════════════════════════════════════════════════════
# 9. ABSORBER OPTIMIZATION (Den Hartog + paper method)
# ═══════════════════════════════════════════════════════

def den_hartog_absorber(z_cav_start):
    """Den Hartog tuning for primary wn1."""
    md, _ = absorber_params(z_cav_start)
    M_mat, _, K_mat, n_free, _, _ = build_FEM_matrices(z_cav_start)
    n = M_mat.shape[0]//2
    evals, _ = eigh(K_mat[:n,:n], M_mat[:n,:n], subset_by_index=[0,0])
    wn1 = np.sqrt(max(evals[0],1.0))
    Mm_modal = 1.0   # normalized (we use tip displacement scaling)
    mu = md / (rho_body * np.pi*(D_main/2)**2 * L_total)  # rough mass ratio
    mu = max(mu, 0.01)
    f_opt = 1.0/(1+mu)
    zeta_opt = np.sqrt(3*mu/(8*(1+mu)**3))
    wd = f_opt * wn1
    kd = md * wd**2
    cd = 2 * zeta_opt * md * wd
    return kd, cd, md

def optimize_absorber_boring(z_cav_start, n_speeds=7):
    """Maximize mean stability limit over speed range."""
    md, _ = absorber_params(z_cav_start)
    speeds = np.linspace(0, RPM_max, n_speeds)
    freqs_stab = np.linspace(10, 800, 400)

    # Get wn1 for initial guess
    M0, _, K0, nf, _, _ = build_FEM_matrices(z_cav_start)
    n0 = M0.shape[0]//2
    ev0, _ = eigh(K0[:n0,:n0], M0[:n0,:n0], subset_by_index=[0,0])
    wn1 = np.sqrt(max(ev0[0], 1.0))

    def neg_mean_alim(log_params):
        kd_ = 10**log_params[0]; cd_ = 10**log_params[1]
        if kd_ < 1e4 or kd_ > 1e8 or cd_ < 1 or cd_ > 5000:
            return 1e10
        total = 0.0
        for rpm in speeds:
            Om = rpm*2*np.pi/60
            Hxx, Hxy = compute_FRF_FEM(freqs_stab, z_cav_start, Om, True, kd_, cd_)
            total += stability_limit_FRF(Hxx, Hxy, freqs_stab)
        return -np.sqrt(total/len(speeds))

    # Den Hartog as starting point
    kd_dh, cd_dh, _ = den_hartog_absorber(z_cav_start)
    x0 = [np.log10(kd_dh), np.log10(cd_dh)]
    res = minimize_scalar(lambda lkd: neg_mean_alim([lkd, x0[1]]),
                          bounds=(4, 8), method='bounded',
                          options={'xatol':0.1,'maxiter':30})
    kd_opt = 10**res.x
    res2 = minimize_scalar(lambda lcd: neg_mean_alim([np.log10(kd_opt), lcd]),
                           bounds=(0, np.log10(5000)), method='bounded',
                           options={'xatol':0.1,'maxiter':30})
    cd_opt = 10**res2.x
    return kd_opt, cd_opt, md


In [ ]:
# ═══════════════════════════════════════════════════════
# 10. OBJECTIVE FUNCTIONS
# ═══════════════════════════════════════════════════════

def compute_all_objectives(z_cav_start, verbose=False):
    """
    Returns dict with:
      - alim_mean: mean stability limit over speed range (maximize)
      - peak_receptance_inv: 1/peak_FRF_magnitude at zero speed (maximize = minimize peak)
      - static_stiffness: F/delta at tip under unit static load (maximize)
      - bar_mass: total mass (minimize → return negative for combined obj)
      - fn1: first natural frequency
    """
    md, z_d = absorber_params(z_cav_start)
    kd_opt, cd_opt, _ = optimize_absorber_boring(z_cav_start)

    # --- Static stiffness: K_tip = 1/H(0) ---
    M0, C0, K0, nf, _, _ = build_FEM_matrices(z_cav_start, 0, True, kd_opt, cd_opt)
    tip_dof = 2*(N_FEM-1)
    f_static = np.zeros(M0.shape[0]); f_static[tip_dof] = 1.0
    try:
        u_static = np.linalg.solve(K0, f_static)
        static_stiff = 1.0 / abs(u_static[tip_dof])
    except:
        static_stiff = 0.0

    # --- Peak receptance ---
    freqs_frf = np.linspace(5, 800, 500)
    Hxx0, Hxy0 = compute_FRF_FEM(freqs_frf, z_cav_start, 0, True, kd_opt, cd_opt)
    peak_H = np.max(np.abs(Hxx0))
    peak_rec_inv = 1.0 / (peak_H + 1e-20)

    # --- Mean stability limit ---
    speeds = np.linspace(0, RPM_max, 9)
    freqs_stab = np.linspace(10, 800, 400)
    alim_vals = []
    for rpm in speeds:
        Om = rpm*2*np.pi/60
        Hxx, Hxy = compute_FRF_FEM(freqs_stab, z_cav_start, Om, True, kd_opt, cd_opt)
        alim_vals.append(stability_limit_FRF(Hxx, Hxy, freqs_stab))
    alim_mean = np.mean(alim_vals) * 1000  # mm

    # --- Bar mass ---
    zz = np.linspace(0, L_total, 2000)
    mass_bar = 0.0
    for i in range(len(zz)-1):
        dz = zz[i+1]-zz[i]; zm = (zz[i]+zz[i+1])/2
        A,_,_,_,_,_ = section_props(zm, z_cav_start)
        mass_bar += rho_body * A * dz
    mass_bar += md
    mass_bar_neg = -mass_bar  # negative for "minimize mass" objective

    # --- fn1 ---
    n = M0.shape[0]//2
    evals, _ = eigh(K0[:n,:n], M0[:n,:n], subset_by_index=[0,0])
    fn1 = np.sqrt(max(evals[0],1.0))/(2*np.pi)

    if verbose:
        print(f"  z_cav={z_cav_start*1e3:.1f}mm | md={md*1e3:.1f}g | z_d={z_d*1e3:.1f}mm")
        print(f"  kd={kd_opt:.2e} N/m, cd={cd_opt:.1f} Ns/m")
        print(f"  fn1={fn1:.1f}Hz | alim_mean={alim_mean:.3f}mm")
        print(f"  static_stiff={static_stiff/1e6:.2f} MN/m | peak_H={peak_H*1e6:.3f} µm/N")
        print(f"  mass={mass_bar:.3f} kg")

    return dict(alim_mean=alim_mean, peak_rec_inv=peak_rec_inv,
                static_stiff=static_stiff, mass_neg=mass_bar_neg,
                fn1=fn1, kd=kd_opt, cd=cd_opt, md=md, z_d=z_d,
                mass=mass_bar)

In [ ]:
# ═══════════════════════════════════════════════════════
# 11. SWEEP CAVITY POSITION
# ═══════════════════════════════════════════════════════

# Valid range: z_cav_start must satisfy:
#   z_cav_start >= L1+L2  (can't be in taper)
#   z_cav_start + L4 <= L_total - L5_min (must leave room for Sec5)
L5_min = 0.030  # minimum solid tip section
z_cav_min = L1 + L2 + 0.005        # just after taper
z_cav_max = L_total - L4 - L5_min  # leave 30mm solid tip

# Coarse sweep
n_sweep = 10
z_sweep = np.linspace(z_cav_min, z_cav_max, n_sweep)

print(f"\nCavity position sweep: {z_cav_min*1e3:.1f}→{z_cav_max*1e3:.1f} mm "
      f"({n_sweep} points)")
print("-"*55)

results = []
for i, z in enumerate(z_sweep):
    print(f"[{i+1}/{n_sweep}] z_cav = {z*1e3:.1f} mm ...", end=' ', flush=True)
    try:
        r = compute_all_objectives(z, verbose=True)
        r['z_cav'] = z
        results.append(r)
    except Exception as ex:
        print(f"  ERROR: {ex}")

print(f"\nSweep complete: {len(results)} valid points")

In [ ]:
# ═══════════════════════════════════════════════════════
# 12. FIND BEST POSITION PER OBJECTIVE
# ═══════════════════════════════════════════════════════

if results:
    z_arr    = np.array([r['z_cav']       for r in results])*1e3
    alim_arr = np.array([r['alim_mean']   for r in results])
    stiff_arr= np.array([r['static_stiff']for r in results])/1e6
    peak_arr = np.array([r['peak_rec_inv']for r in results])
    mass_arr = np.array([r['mass']        for r in results])
    fn1_arr  = np.array([r['fn1']         for r in results])

    i_alim  = np.argmax(alim_arr)
    i_stiff = np.argmax(stiff_arr)
    i_peak  = np.argmax(peak_arr)
    i_mass  = np.argmin(mass_arr)

    # Combined (normalized, equal weights)
    def norm01(x): return (x-x.min())/(x.max()-x.min()+1e-30)
    combined = (norm01(alim_arr) + norm01(stiff_arr) +
                norm01(peak_arr) + norm01(-mass_arr)) / 4
    i_best = np.argmax(combined)

    print("\n" + "="*55)
    print("OPTIMAL CAVITY POSITIONS")
    print("="*55)
    print(f"Max stability (alim):    z = {z_arr[i_alim]:.1f} mm  "
          f"→ alim = {alim_arr[i_alim]:.3f} mm")
    print(f"Max static stiffness:    z = {z_arr[i_stiff]:.1f} mm  "
          f"→ K = {stiff_arr[i_stiff]:.2f} MN/m")
    print(f"Min peak receptance:     z = {z_arr[i_peak]:.1f} mm  "
          f"→ 1/H_peak = {peak_arr[i_peak]:.2e}")
    print(f"Min mass:                z = {z_arr[i_mass]:.1f} mm  "
          f"→ m = {mass_arr[i_mass]*1e3:.0f} g")
    print(f"\n★ BEST COMBINED:         z = {z_arr[i_best]:.1f} mm")
    best = results[i_best]
    print(f"   alim={best['alim_mean']:.3f}mm | K_static={best['static_stiff']/1e6:.2f}MN/m")
    print(f"   kd={best['kd']:.2e} N/m | cd={best['cd']:.1f} Ns/m | md={best['md']*1e3:.0f}g")
    print("="*55)


In [ ]:
# ═══════════════════════════════════════════════════
    # 13. DETAILED PLOTS FOR BEST POSITION
    # ═══════════════════════════════════════════════════
    z_best = results[i_best]['z_cav']
    kd_best= results[i_best]['kd']
    cd_best= results[i_best]['cd']
    md_best= results[i_best]['md']

    fig = plt.figure(figsize=(15, 13))
    fig.suptitle(f'Boring Bar Optimization — Best cavity position: z = {z_best*1e3:.1f} mm\n'
                 f'kd={kd_best:.2e} N/m, cd={cd_best:.0f} Ns/m, md={md_best*1e3:.0f} g',
                 fontsize=11, fontweight='bold')
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # ── A: Objective functions vs cavity position ──
    ax_a1 = fig.add_subplot(gs[0,0])
    ax_a2 = fig.add_subplot(gs[0,1])
    ax_a3 = fig.add_subplot(gs[0,2])

    ax_a1.plot(z_arr, alim_arr, 'b-o', ms=5, lw=1.5)
    ax_a1.axvline(z_arr[i_best], color='r', ls='--', lw=1.2, label='Best combined')
    ax_a1.axvline(z_arr[i_alim], color='g', ls=':', lw=1.2, label='Max alim')
    ax_a1.set_xlabel('Cavity start z [mm]'); ax_a1.set_ylabel('Mean alim [mm]')
    ax_a1.set_title('(a) Stability limit vs cavity pos.'); ax_a1.legend(fontsize=7); ax_a1.grid(True, alpha=0.3)

    ax_a2.plot(z_arr, stiff_arr, 'g-o', ms=5, lw=1.5)
    ax_a2.axvline(z_arr[i_best], color='r', ls='--', lw=1.2)
    ax_a2.axvline(z_arr[i_stiff], color='g', ls=':', lw=1.2)
    ax_a2.set_xlabel('Cavity start z [mm]'); ax_a2.set_ylabel('Static stiffness [MN/m]')
    ax_a2.set_title('(b) Static stiffness vs cavity pos.'); ax_a2.grid(True, alpha=0.3)

    ax_a3.plot(z_arr, mass_arr*1e3, 'm-o', ms=5, lw=1.5)
    ax_a3.axvline(z_arr[i_best], color='r', ls='--', lw=1.2)
    ax_a3.axvline(z_arr[i_mass], color='m', ls=':', lw=1.2)
    ax_a3.set_xlabel('Cavity start z [mm]'); ax_a3.set_ylabel('Total mass [g]')
    ax_a3.set_title('(c) Bar mass vs cavity pos.'); ax_a3.grid(True, alpha=0.3)

    # ── B: Cross-section profile ──
    ax_b = fig.add_subplot(gs[1,:])
    zz_plot = np.linspace(0, L_total, 1000)
    R_out_plot = []; R_in_plot = []
    for zp in zz_plot:
        A,I,Ip,_,_,_ = section_props(zp, z_best)
        R_o = np.sqrt(A/np.pi + (D_body_in/2)**2) if zp > L1+L2 else D1_out/2  # approx
        # Direct calculation
        z1=L1; z2=L1+L2; z3=z_best; z4=z_best+L4
        if zp < z1: D_o=D1_out
        elif zp < z2: D_o=D1_out+(zp-z1)/L2*(D_main-D1_out)
        else: D_o=D_main
        R_o = D_o/2
        if zp < z2: R_i = max(0, R_o - d_body_wall)
        elif zp < z3: R_i = D_body_in/2
        elif zp < z4: R_i = R_o - d_body_wall
        else: R_i = 0.0
        R_out_plot.append(R_o*1e3); R_in_plot.append(R_i*1e3)

    ax_b.fill_between(zz_plot*1e3,  np.array(R_out_plot),  np.array(R_in_plot),
                      alpha=0.6, color='steelblue', label='Body (WC-composite)')
    ax_b.fill_between(zz_plot*1e3, -np.array(R_in_plot), -np.array(R_out_plot),
                      alpha=0.6, color='steelblue')
    ax_b.fill_between(zz_plot*1e3,  np.array(R_in_plot), -np.array(R_in_plot),
                      alpha=0.15, color='lightblue', label='Bore/cavity')
    # Absorber
    _, z_d_b = absorber_params(z_best)
    R_cav = D_main/2 - d_body_wall
    ax_b.fill_betweenx([-R_cav*1e3*0.85, R_cav*1e3*0.85],
                        z_best*1e3, (z_best+L4)*1e3,
                        alpha=0.7, color='darkorange', label=f'Absorber (TC) md={md_best*1e3:.0f}g')
    ax_b.axvline(z_best*1e3, color='r', ls='--', lw=1.2, label=f'Cavity start z={z_best*1e3:.1f}mm')
    ax_b.axvline((z_best+L4)*1e3, color='r', ls=':', lw=1.2)
    ax_b.set_xlim(0, L_total*1e3); ax_b.set_ylim(-35,35)
    ax_b.set_xlabel('Axial position z [mm]'); ax_b.set_ylabel('Radius [mm]')
    ax_b.set_title('(d) Cross-section profile — optimal cavity position', fontsize=9)
    ax_b.legend(fontsize=7, loc='upper right'); ax_b.grid(True, alpha=0.3)
    # Section labels
    for zl, lbl in [(L1/2, 'Sec1'), (L1+L2/2, 'Sec2\n(taper)'),
                    (L1+L2+(z_best-L1-L2)/2, 'Sec3'),
                    (z_best+L4/2, 'Sec4\nabsorber'),
                    (z_best+L4+(L_total-z_best-L4)/2, 'Sec5')]:
        ax_b.text(zl*1e3, 28, lbl, ha='center', fontsize=7, color='navy')

    # ── C: FRF at best position ──
    ax_c1 = fig.add_subplot(gs[2,0])
    ax_c2 = fig.add_subplot(gs[2,1])

    freqs_frf = np.linspace(5, 800, 600)
    speeds_plot = [0, 1000, 3000, 6000]
    clrs = ['k','b','g','r']
    for rpm, c in zip(speeds_plot, clrs):
        Om = rpm*2*np.pi/60
        Hxx, Hxy = compute_FRF_FEM(freqs_frf, z_best, Om, True, kd_best, cd_best)
        ax_c1.semilogy(freqs_frf, np.clip(np.abs(Hxx),1e-12,None), color=c, lw=1.2, label=f'{rpm} RPM')
        ax_c2.semilogy(freqs_frf, np.clip(np.abs(Hxy),1e-12,None), color=c, lw=1.2, label=f'{rpm} RPM')
    for ax, t in [(ax_c1,'(e) Direct FRF H_xx'),(ax_c2,'(f) Cross FRF H_xy')]:
        ax.set_xlabel('Freq [Hz]'); ax.set_ylabel('|H| [m/N]')
        ax.set_title(t); ax.legend(fontsize=7); ax.grid(True,which='both',alpha=0.3)
        ax.set_xlim(5, 800)

    # ── D: Combined objective score ──
    ax_d = fig.add_subplot(gs[2,2])
    ax_d.plot(z_arr, combined*100, 'k-o', ms=5, lw=1.8)
    ax_d.axvline(z_arr[i_best], color='r', ls='--', lw=1.5,
                 label=f'Best: z={z_arr[i_best]:.1f}mm')
    ax_d.fill_between(z_arr, combined*100, alpha=0.2, color='tomato')
    ax_d.set_xlabel('Cavity start z [mm]'); ax_d.set_ylabel('Combined score [%]')
    ax_d.set_title('(g) Combined objective score'); ax_d.legend(fontsize=8)
    ax_d.grid(True, alpha=0.3)

    plt.savefig('/mnt/user-data/outputs/boring_bar_optimization.png',
                bbox_inches='tight', dpi=150)
    print("\nFigure saved.")

print("\n=== DONE ===")